In [10]:
import json, random, difflib
from mlx_lm import load, generate
from apriori_core import load_json, mine, predict

MODEL, ADAPTER = "Qwen/Qwen2.5-0.5B-Instruct", "out_mlx"      # change ADAPTER if you renamed the folder
FIELDS = ["category", "labels", "shelf_life_days", "storage", "common_uses"]
INSTRUCTION = "Suggest the best item to add to this cart. Do not generate the outcome on your own"

ds2 = {d["item_id"]: d for d in load_json("ds2_descriptions.json")}
ID = {d["name"]: i for i, d in ds2.items()}
rules, _ = mine(load_json("ds1_train.json"), 0.004, 0.3)
model, tok = load(MODEL, adapter_path=ADAPTER)

def describe(i):
    d = ds2[i]; parts = [i, d["name"]]
    for f in FIELDS:
        v = d[f]; parts.append(f"{f}: {', '.join(map(str, v)) if isinstance(v, list) else v}")
    return " | ".join(parts)

def build_prompt(*names):
    ids = []
    for n in names:
        if n not in ID: raise ValueError(f"'{n}' not found. Did you mean: {difflib.get_close_matches(n, ID, 3)}")
        ids.append(ID[n])
    hits = predict(rules, ids, k=4)
    cats = list(dict.fromkeys(ds2[h[0]]["category"] for h in hits))
    cand = list(dict.fromkeys(i for c in cats for i, d in ds2.items() if d["category"] == c))[:12]
    random.Random(0).shuffle(cand)
    rule_txt = "\n".join(f"{' + '.join(sorted(ds2[i]['name'] for i in h[3]))} => {ds2[h[0]]['name']}" for h in hits) or "(no rule matched)"
    prompt = (INSTRUCTION + "\n\nCART:\n" + "\n".join(describe(i) for i in ids) + f"\n\nMINED RULES:\n{rule_txt}\n\n"
              + ("CANDIDATES:\n" + "\n".join(describe(i) for i in cand) + "\n\n" if cand else "") + "ANSWER:")
    return prompt, hits

def ask(*names, show_prompt=False):
    prompt, hits = build_prompt(*names)
    if getattr(tok, "chat_template", None):
        text = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
    else:
        text = prompt
    out = generate(model, tok, prompt=text, max_tokens=20).strip()
    if show_prompt: print(prompt, "\n")
    print("CART      :", " + ".join(names))
    print("APRIORI   :", ", ".join(f"{ds2[h[0]]['name']} ({h[1]:.2f})" for h in hits[:3]) or "-- no rule --")
    print("FINE-TUNED:", out, "\n")

# ---- one-off tests: edit freely (names must match DS2 exactly) ----
ask("Whole Milk", "Rye Bread")                # expect Fat-Free Butter

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 2229.15it/s]


CART      : Whole Milk + Rye Bread
APRIORI   : Salted Butter (1.00), Vegan Butter (0.34), Free Range Eggs (0.34)
FINE-TUNED: OB_1059 | Free Range Eggs 



In [4]:
# ---- Test missed predictions ----

# 1. Expected: Vegan Sausages
ask("Vegan Burger Patties", "Vegan Hot Dog Buns")

# 2. Expected: Veggie Sausages
ask("Vegan Burger Patties", "Hot Dog Buns")

# 3. Expected: Vegan Mayo
ask("Vegan Cheese Slices", "Bananas", "Sourdough")

# 4. Expected: Coconut Yogurt
ask("Gluten-Free Muesli", "Almond Milk")

# 5. Expected: Vegan Butter
ask("Sourdough", "Soy Milk")

# 6. Expected: Lactose-Free Yogurt
ask("Corn Flakes", "Lactose-Free Milk", "Basmati Rice")

# 7. Expected: Egg Whites Carton
ask("Skimmed Milk", "Plain Flour")

# 8. Expected: Vegetarian Hard Cheese
ask("Vegan Bolognese Sauce", "Egg Tagliatelle", "Dishwasher Tablets")

CART      : Vegan Burger Patties + Vegan Hot Dog Buns
APRIORI   : Pork Sausages (0.60), Vegan Sausages (0.40)
FINE-TUNED: OB_1073 | Pork Sausages 

CART      : Vegan Burger Patties + Hot Dog Buns
APRIORI   : Pork Sausages (1.00)
FINE-TUNED: OB_1073 | Pork Sausages 

CART      : Vegan Cheese Slices + Bananas + Sourdough
APRIORI   : Mayonnaise (0.47), Vegan Mayo (0.37), Eggless Mayo (0.31)
FINE-TUNED: OB_1052 | Mayonnaise 

CART      : Gluten-Free Muesli + Almond Milk
APRIORI   : Greek Yogurt (0.33), Coconut Yogurt (0.32)
FINE-TUNED: OB_1025 | Coconut Yogurt 

CART      : Sourdough + Soy Milk
APRIORI   : Mayonnaise (0.47)
FINE-TUNED: OB_1052 | Mayonnaise 

CART      : Corn Flakes + Lactose-Free Milk + Basmati Rice
APRIORI   : Coconut Yogurt (0.58), Lactose-Free Yogurt (0.50), Low Fat Yogurt (0.39)
FINE-TUNED: OB_1025 | Coconut Yogurt 

CART      : Skimmed Milk + Plain Flour
APRIORI   : Free Range Eggs (0.61), Egg Whites Carton (0.39), Fat-Free Butter (0.37)
FINE-TUNED: OB_1059 | Free Ran

In [10]:
## Test example Vegan Burger Patties + Vegan Hot Dog Buns

text = """
Suggest the best item to add to this cart. Do not generate the outcome on your own, check the context of the cart items before you suggest.
cart = Vegan Burger Patties, Vegan Hot Dog Buns
apriori = Pork Sausages (0.60), Vegan Sausages (0.40)
"""

prompt = tok.apply_chat_template(
    [{"role": "user", "content": text}],
    add_generation_prompt=True,
    tokenize=False
)

out = generate(model, tok, prompt=prompt, max_tokens=100).strip()

print(out)

OB S S
!
OB
03arian Sarian ReOB S S03 | Maylessarian Mayoplaceronna Range May

! | Range Mayless Sarian May Rangearian Sarian
! |OB Vegan Yogausless Sausages


In [9]:

from mlx_lm import generate

text = """Suggest the best item to add to this cart. Do not generate the outcome on your own, check the context of the cart items before you suggest.

CART:
Vegan Burger Patties + Vegan Hot Dog Buns

APRIORI:
Pork Sausages (0.60)
Vegan Sausages (0.40)

ANSWER:"""

prompt = tok.apply_chat_template(
    [{"role": "user", "content": text}],
    add_generation_prompt=True,
    tokenize=False
)

out = generate(model, tok, prompt=prompt, max_tokens=32).strip()

print("FINE-TUNED:", out)


FINE-TUNED: OB
003 Veganarian Re Rangearianarian Hard Mayesanplacer Sarian

OB May |003 MayOB May0000


In [5]:

import random
import difflib
import re
from mlx_lm import load, generate
from apriori_core import load_json, mine, predict

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER = "out_mlx"

FIELDS = [
    "category", "labels", "shelf_life_days",
    "storage", "common_uses"
]

# Match the original training instruction
INSTRUCTION = "Suggest the best item to add to this cart."

ds2 = {d["item_id"]: d for d in load_json("ds2_descriptions.json")}
ID = {d["name"]: i for i, d in ds2.items()}

rules, _ = mine(load_json("ds1_train.json"), 0.004, 0.3)
model, tok = load(MODEL, adapter_path=ADAPTER)


def describe(i):
    d = ds2[i]
    parts = [i, d["name"]]

    for f in FIELDS:
        v = d[f]
        value = ", ".join(map(str, v)) if isinstance(v, list) else v
        parts.append(f"{f}: {value}")

    return " | ".join(parts)


def build_prompt(*names, include_confidence=False):
    ids = []

    for n in names:
        if n not in ID:
            raise ValueError(
                f"'{n}' not found. Did you mean: "
                f"{difflib.get_close_matches(n, ID, 3)}"
            )
        ids.append(ID[n])

    hits = predict(rules, ids, k=4)

    # Original category expansion, preserving the existing
    # inference approach for a fair comparison.
    cats = list(dict.fromkeys(
        ds2[h[0]]["category"] for h in hits
    ))

    cand = list(dict.fromkeys(
        i
        for c in cats
        for i, d in ds2.items()
        if d["category"] == c and i not in ids
    ))[:12]

    random.Random(0).shuffle(cand)

    rule_lines = []

    for h in hits:
        antecedent = " + ".join(
            sorted(ds2[i]["name"] for i in h[3])
        )
        consequent = ds2[h[0]]["name"]

        line = f"{antecedent} => {consequent}"

        if include_confidence:
            line += f" | confidence: {h[1]:.3f}"

        rule_lines.append(line)

    rule_txt = "\n".join(rule_lines) or "(no rule matched)"

    prompt = (
        INSTRUCTION
        + "\n\nCART:\n"
        + "\n".join(describe(i) for i in ids)
        + "\n\nMINED RULES:\n"
        + rule_txt
        + "\n\n"
        + (
            "CANDIDATES:\n"
            + "\n".join(describe(i) for i in cand)
            + "\n\n"
            if cand else ""
        )
        + "ANSWER:"
    )

    return prompt, hits, cand


def ask(
    *names,
    expected=None,
    show_prompt=False,
    include_confidence=False
):
    prompt, hits, candidates = build_prompt(
        *names,
        include_confidence=include_confidence
    )

    if getattr(tok, "chat_template", None):
        formatted = tok.apply_chat_template(
            [{"role": "user", "content": prompt}],
            add_generation_prompt=True,
            tokenize=False
        )
    else:
        formatted = prompt

    out = generate(
        model,
        tok,
        prompt=formatted,
        max_tokens=20
    ).strip()

    # Extract generated product ID
    match = re.search(r"\bOB_\d+\b", out)
    predicted_id = match.group(0) if match else None

    apriori_id = hits[0][0] if hits else None
    expected_id = ID.get(expected) if expected else None

    # Check that ID and generated name agree
    canonical_name = (
        ds2[predicted_id]["name"]
        if predicted_id in ds2 else None
    )

    generated_name = (
        out.split("|", 1)[1].strip()
        if "|" in out else None
    )

    id_name_match = (
        generated_name == canonical_name
        if canonical_name and generated_name
        else None
    )

    print("\n" + "=" * 65)
    print("CART:", " + ".join(names))

    print(
        "APRIORI:",
        ", ".join(
            f"{ds2[h[0]]['name']} ({h[1]:.2f})"
            for h in hits[:3]
        ) or "-- no rule --"
    )

    print("FINE-TUNED:", out)
    print("PREDICTED ID:", predicted_id)
    print("CANONICAL NAME:", canonical_name)
    print("VALID PRODUCT ID:", predicted_id in ds2)
    print("ID/NAME MATCH:", id_name_match)
    print("IN CANDIDATES:", predicted_id in candidates)
    print("FOLLOWS APRIORI TOP-1:", predicted_id == apriori_id)

    if expected is not None:
        if expected_id is None:
            raise ValueError(f"Unknown expected product: {expected}")

        print("EXPECTED:", expected)
        print("EXPECTED IN CANDIDATES:",
              expected_id in candidates)
        print("MODEL CORRECT:", predicted_id == expected_id)
        print("APRIORI CORRECT:", apriori_id == expected_id)

    if show_prompt:
        print("\nFULL PROMPT:\n")
        print(prompt)

    print("=" * 65)

    return {
        "cart": names,
        "expected": expected,
        "predicted_id": predicted_id,
        "predicted_name": canonical_name,
        "apriori_top1": apriori_id,
        "candidates": candidates,
        "model_correct": (
            predicted_id == expected_id
            if expected is not None else None
        ),
        "apriori_correct": (
            apriori_id == expected_id
            if expected is not None else None
        ),
        "expected_in_candidates": (
            expected_id in candidates
            if expected is not None else None
        ),
        "follows_apriori": predicted_id == apriori_id,
        "id_name_match": id_name_match,
        "raw_output": out
    }


Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 1955.26it/s]


In [6]:

tests = [
    (("Vegan Burger Patties", "Vegan Hot Dog Buns"), "Vegan Sausages"),
    (("Vegan Burger Patties", "Hot Dog Buns"), "Veggie Sausages"),
    (("Vegan Cheese Slices", "Bananas", "Sourdough"), "Vegan Mayo"),
    (("Gluten-Free Muesli", "Almond Milk"), "Coconut Yogurt"),
    (("Sourdough", "Soy Milk"), "Vegan Butter"),
    (("Corn Flakes", "Lactose-Free Milk", "Basmati Rice"), "Lactose-Free Yogurt"),
    (("Skimmed Milk", "Plain Flour"), "Egg Whites Carton"),
    (("Vegan Bolognese Sauce", "Egg Tagliatelle", "Dishwasher Tablets"), "Vegetarian Hard Cheese"),
]

results = []

for cart, expected in tests:
    result = ask(
        *cart,
        expected=expected,
        include_confidence=False
    )
    results.append(result)

n = len(results)

print("\n===== SUMMARY =====")
print("MODEL HIT@1:", sum(r["model_correct"] for r in results), "/", n)
print("APRIORI HIT@1:", sum(r["apriori_correct"] for r in results), "/", n)
print("EXPECTED IN CANDIDATES:",
      sum(r["expected_in_candidates"] for r in results), "/", n)
print("MODEL FOLLOWS APRIORI:",
      sum(r["follows_apriori"] for r in results), "/", n)
print("MODEL OUTPUT IN CANDIDATES:",
      sum(r["predicted_id"] in r["candidates"] for r in results), "/", n)
print("ID/NAME MISMATCHES:",
      sum(r["id_name_match"] is False for r in results), "/", n)



CART: Vegan Burger Patties + Vegan Hot Dog Buns
APRIORI: Pork Sausages (0.60), Vegan Sausages (0.40)
FINE-TUNED: OB_1073 | Pork Sausages
PREDICTED ID: OB_1073
CANONICAL NAME: Pork Sausages
VALID PRODUCT ID: True
ID/NAME MATCH: True
IN CANDIDATES: True
FOLLOWS APRIORI TOP-1: True
EXPECTED: Vegan Sausages
EXPECTED IN CANDIDATES: True
MODEL CORRECT: False
APRIORI CORRECT: False

CART: Vegan Burger Patties + Hot Dog Buns
APRIORI: Pork Sausages (1.00)
FINE-TUNED: OB_1073 | Pork Sausages
PREDICTED ID: OB_1073
CANONICAL NAME: Pork Sausages
VALID PRODUCT ID: True
ID/NAME MATCH: True
IN CANDIDATES: True
FOLLOWS APRIORI TOP-1: True
EXPECTED: Veggie Sausages
EXPECTED IN CANDIDATES: True
MODEL CORRECT: False
APRIORI CORRECT: False

CART: Vegan Cheese Slices + Bananas + Sourdough
APRIORI: Mayonnaise (0.47), Vegan Mayo (0.37), Eggless Mayo (0.31)
FINE-TUNED: OB_1052 | Mayonnaise
PREDICTED ID: OB_1052
CANONICAL NAME: Mayonnaise
VALID PRODUCT ID: True
ID/NAME MATCH: True
IN CANDIDATES: True
FOLLOWS A

In [7]:

confidence_results = []

for cart, expected in tests:
    result = ask(
        *cart,
        expected=expected,
        include_confidence=True
    )
    confidence_results.append(result)

print("\n===== COMPARISON =====")

print(
    "WITHOUT CONFIDENCE:",
    sum(r["model_correct"] for r in results),
    "/", len(results)
)

print(
    "WITH CONFIDENCE:",
    sum(r["model_correct"] for r in confidence_results),
    "/", len(confidence_results)
)



CART: Vegan Burger Patties + Vegan Hot Dog Buns
APRIORI: Pork Sausages (0.60), Vegan Sausages (0.40)
FINE-TUNED: OB_1073 | Pork Sausages
PREDICTED ID: OB_1073
CANONICAL NAME: Pork Sausages
VALID PRODUCT ID: True
ID/NAME MATCH: True
IN CANDIDATES: True
FOLLOWS APRIORI TOP-1: True
EXPECTED: Vegan Sausages
EXPECTED IN CANDIDATES: True
MODEL CORRECT: False
APRIORI CORRECT: False

CART: Vegan Burger Patties + Hot Dog Buns
APRIORI: Pork Sausages (1.00)
FINE-TUNED: OB_1073 | Pork Sausages
PREDICTED ID: OB_1073
CANONICAL NAME: Pork Sausages
VALID PRODUCT ID: True
ID/NAME MATCH: True
IN CANDIDATES: True
FOLLOWS APRIORI TOP-1: True
EXPECTED: Veggie Sausages
EXPECTED IN CANDIDATES: True
MODEL CORRECT: False
APRIORI CORRECT: False

CART: Vegan Cheese Slices + Bananas + Sourdough
APRIORI: Mayonnaise (0.47), Vegan Mayo (0.37), Eggless Mayo (0.31)
FINE-TUNED: OB_1052 | Mayonnaise
PREDICTED ID: OB_1052
CANONICAL NAME: Mayonnaise
VALID PRODUCT ID: True
ID/NAME MATCH: True
IN CANDIDATES: True
FOLLOWS A

In [8]:

import json
import re
from collections import Counter

path = "comparison_400/predictions.jsonl"

with open(path) as f:
    rows = [json.loads(line) for line in f]

print("FIRST 15 BASE MODEL OUTPUTS:\n")

for r in rows[:15]:
    print("Expected:", r["expected_id"])
    print("Base raw:", repr(r["base_raw"]))
    print("Fine-tuned raw:", repr(r["finetuned_raw"]))
    print("-" * 60)

# Diagnostic only: find IDs anywhere in the output
base_ids_anywhere = [
    re.search(r"\bOB_\d+\b", r["base_raw"])
    for r in rows
]

print("\nBase outputs containing an ID anywhere:",
      sum(m is not None for m in base_ids_anywhere),
      "/", len(rows))

print("\nMost common base outputs:")
for output, count in Counter(
    r["base_raw"] for r in rows
).most_common(10):
    print(count, repr(output))


FIRST 15 BASE MODEL OUTPUTS:

Expected: OB_1015
Base raw: 'Based on the information provided in the cart, the best item for this cart would be OB_1001,'
Fine-tuned raw: 'OB_1059 | Free Range Eggs'
------------------------------------------------------------
Expected: OB_1015
Base raw: 'Based on the information provided in the cart, the best item for this cart would be OB_1007,'
Fine-tuned raw: 'OB_1060 | Egg Replacer'
------------------------------------------------------------
Expected: OB_1075
Base raw: 'Based on the information provided in the cart, the best item for this cart would be:\n\nOB_1072'
Fine-tuned raw: 'OB_1073 | Vegangie Sages'
------------------------------------------------------------
Expected: OB_1074
Base raw: 'Based on the information provided in the cart, the best item for this cart would be OB_1064:'
Fine-tuned raw: 'OB_1073 | Veggie Sausages'
------------------------------------------------------------
Expected: OB_1052
Base raw: 'Based on the information provi